# K-Nearest Neighbors on the Diabetes Dataset
**Task:** Implement the KNN algorithm on `diabetes.csv` (Pima Indians Diabetes) and compute the **confusion matrix, accuracy, error rate, precision and recall**.

`Outcome = 1` means the patient has diabetes (positive class), `0` means no diabetes.

> Keep `diabetes.csv` in the same folder as this notebook.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics

sns.set_style("whitegrid")
RANDOM_STATE = 42

## 2. Load and explore the data

In [ ]:
diabetes_data = pd.read_csv("diabetes.csv")
print("Shape:", diabetes_data.shape)
diabetes_data.head()

In [ ]:
diabetes_data.info()

In [ ]:
diabetes_data.describe().T

### Hidden missing values
In this dataset a value of **0** is impossible for `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` and `BMI` - these zeros actually represent *missing* measurements. (`Pregnancies = 0` is valid.) We convert them to `NaN`.

In [ ]:
cols_with_hidden_nan = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

df = diabetes_data.copy()
df[cols_with_hidden_nan] = df[cols_with_hidden_nan].replace(0, np.nan)

missing = df.isnull().sum().to_frame("Missing values")
missing["% missing"] = (100 * missing["Missing values"] / len(df)).round(2)
missing

### Class balance

In [ ]:
print(df["Outcome"].value_counts())
sns.countplot(x="Outcome", data=df, palette=["#2ca02c", "#d62728"])
plt.xticks([0, 1], ["0 - No diabetes", "1 - Diabetes"])
plt.title("Class distribution")
plt.show()

### Feature distributions

In [ ]:
df.drop(columns="Outcome").hist(figsize=(14, 9), bins=25, color="steelblue")
plt.tight_layout()
plt.show()

### Correlation heatmap

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="RdYlGn")
plt.title("Correlation matrix (missing values ignored)")
plt.show()

`Glucose`, `BMI`, `Age` and `Pregnancies` show the strongest relation with `Outcome`.

## 3. Train / test split and pre-processing

* Features `X` and target `y` are split first (stratified 2/3 train, 1/3 test).
* **Imputation** (median) and **scaling** are done inside a pipeline, so they are *learned from the training data only* - no information leaks from the test set. KNN is distance based, so scaling is essential.

In [ ]:
X = df.drop(columns="Outcome")
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=1/3, random_state=RANDOM_STATE, stratify=y)

print("Train:", X_train.shape, " Test:", X_test.shape)

## 4. Choose the best K
We use 5-fold cross-validation on the **training set** (the test set stays untouched until the final evaluation). The train-vs-test curve of the original approach is also shown for understanding under/over-fitting, but K is *selected* using cross-validation.

In [ ]:
def make_knn(k):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         KNeighborsClassifier(n_neighbors=k))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
k_range = range(1, 31)

cv_scores, train_scores, test_scores = [], [], []
for k in k_range:
    model = make_knn(k)
    cv_scores.append(cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy").mean())
    model.fit(X_train, y_train)
    train_scores.append(model.score(X_train, y_train))
    test_scores.append(model.score(X_test, y_test))

best_k = list(k_range)[int(np.argmax(cv_scores))]
print(f"Best K from cross-validation: {best_k}  (CV accuracy = {max(cv_scores):.4f})")

plt.figure(figsize=(11, 5))
plt.plot(k_range, train_scores, marker="*", label="Train accuracy")
plt.plot(k_range, cv_scores, marker="s", label="Cross-validation accuracy")
plt.plot(k_range, test_scores, marker="o", label="Test accuracy", alpha=0.6)
plt.axvline(best_k, color="red", linestyle="--", label=f"best K = {best_k}")
plt.xlabel("K (number of neighbours)"); plt.ylabel("Accuracy")
plt.title("KNN - accuracy vs K"); plt.legend()
plt.show()

## 5. Train the final KNN model

In [ ]:
knn = make_knn(best_k)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
print(f"KNN trained with K = {best_k}")

## 6. Evaluation

For the positive class (diabetes = 1):

| | Predicted 0 | Predicted 1 |
|---|---|---|
| **Actual 0** | TN | FP |
| **Actual 1** | FN | TP |

* **Accuracy** = (TP + TN) / (TP + TN + FP + FN)
* **Error rate** = (FP + FN) / (TP + TN + FP + FN) = 1 − Accuracy
* **Precision** = TP / (TP + FP)
* **Recall** = TP / (TP + FN)

### 6.1 Confusion matrix

In [ ]:
cm = metrics.confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(pd.crosstab(y_test, y_pred, rownames=["Actual"], colnames=["Predicted"], margins=True))
print()
print(f"TN = {tn}, FP = {fp}, FN = {fn}, TP = {tp}")

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu",
            xticklabels=["No diabetes", "Diabetes"], yticklabels=["No diabetes", "Diabetes"])
plt.title(f"Confusion matrix (K = {best_k})")
plt.ylabel("Actual label"); plt.xlabel("Predicted label")
plt.show()

### 6.2 Accuracy, error rate, precision and recall

In [ ]:
# Computed manually from the confusion matrix ...
accuracy_m  = (tp + tn) / (tp + tn + fp + fn)
error_rate  = (fp + fn) / (tp + tn + fp + fn)
precision_m = tp / (tp + fp)
recall_m    = tp / (tp + fn)

# ... and cross-checked with scikit-learn
accuracy_s  = metrics.accuracy_score(y_test, y_pred)
precision_s = metrics.precision_score(y_test, y_pred)
recall_s    = metrics.recall_score(y_test, y_pred)

assert np.isclose(accuracy_m, accuracy_s) and np.isclose(precision_m, precision_s) and np.isclose(recall_m, recall_s)

summary = pd.DataFrame({
    "Metric": ["Accuracy", "Error rate", "Precision", "Recall"],
    "Value": [accuracy_m, error_rate, precision_m, recall_m],
    "Meaning": [
        "Share of all patients classified correctly",
        "Share of all patients classified wrongly (1 - accuracy)",
        "Of those predicted diabetic, how many really are",
        "Of the really diabetic patients, how many were found",
    ],
}).set_index("Metric")
summary["Value"] = summary["Value"].round(4)
summary

In [ ]:
# Full report (also includes the F1-score and the metrics of the negative class)
print(metrics.classification_report(y_test, y_pred, target_names=["No diabetes", "Diabetes"], digits=4))

In [ ]:
summary["Value"].plot(kind="bar", figsize=(7, 4), color=["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e"], rot=0, ylim=(0, 1))
plt.title(f"KNN (K = {best_k}) - test-set metrics")
for p in plt.gca().patches:
    plt.gca().annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2, p.get_height()),
                       ha="center", va="bottom")
plt.show()

### 6.3 Decision boundary (two most informative features)

In [ ]:
# KNN on just Glucose and BMI so the boundary can be drawn in 2-D (visualisation only)
two = ["Glucose", "BMI"]
knn2 = make_knn(best_k).fit(X_train[two], y_train)

gx, gy = np.meshgrid(np.linspace(X[two[0]].min() - 5, X[two[0]].max() + 5, 300),
                     np.linspace(X[two[1]].min() - 3, X[two[1]].max() + 3, 300))
zz = knn2.predict(pd.DataFrame({two[0]: gx.ravel(), two[1]: gy.ravel()})).reshape(gx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(gx, gy, zz, alpha=0.25, cmap="coolwarm")
plt.scatter(X_test[two[0]], X_test[two[1]], c=y_test, cmap="coolwarm", edgecolor="k", s=25)
plt.xlabel(two[0]); plt.ylabel(two[1])
plt.title(f"KNN decision regions (K = {best_k}) - test points shown")
plt.show()

## 7. Predict for a new patient

In [ ]:
new_patient = pd.DataFrame([{
    "Pregnancies": 2, "Glucose": 150, "BloodPressure": 80, "SkinThickness": 30,
    "Insulin": 150, "BMI": 33.0, "DiabetesPedigreeFunction": 0.6, "Age": 45
}])
pred = knn.predict(new_patient)[0]
prob = knn.predict_proba(new_patient)[0, 1]
print("Prediction:", "Diabetic" if pred == 1 else "Not diabetic", f"(probability of diabetes = {prob:.2f})")

## 8. Conclusion

* Zeros in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` and `BMI` were treated as missing values and filled with the median; features were standardised before applying KNN.
* The best **K** was selected with cross-validation, and the final model was evaluated on a held-out test set.
* The **confusion matrix** shows the four outcomes (TN, FP, FN, TP); from it **accuracy**, **error rate**, **precision** and **recall** were computed (and verified against scikit-learn).
* In a medical setting **recall** is especially important: a false negative (a diabetic patient classified as healthy) is more harmful than a false positive. Because the classes are imbalanced, accuracy alone should not be trusted.
* **Possible improvements:** other distance metrics/weights (`weights="distance"`), `GridSearchCV`, SMOTE/class balancing, and comparing with Logistic Regression, SVM or Random Forest.